# Team 19 - Overhead-Conductor Data Pipeline

**Single-notebook interface for the SA Power Networks release handover**

The notebook orchestrates independently testable Python modules in `src/`. It **does not reimplement** ingestion, validation, cleaning, imputation or evaluation: every cell below calls the same stage functions as the `oc-pipeline` command line (`src.pipeline.run_*`), so a run from this notebook and a run from the terminal produce the same files.

Sections: 0 configuration · 1 ingestion · 2 validation · 3 exploratory analysis · 4 preprocessing and deterministic recovery · 5 imputation method · 6 postprocessing and provenance · 7 evaluation and diagnostics · 8 export and report.

> Every file under `data/transformers/` is **synthetic**; no SA Power Networks operational data is read, written or quoted here. Results on synthetic data demonstrate that the pipeline runs end to end and show how the methods behave on a generated population; they do **not** establish performance on real SAPN records. Domain rules that need a SAPN decision (for example the `code_mappings` cleaning family) are shipped **unapproved** and skipped until a domain owner approves them.

## ⚙️ Project Configuration and Dataset Selection

This section contains the central configuration for the capstone data quality and imputation pipeline.

All dataset, asset-specific configuration, and output paths are defined in the code cell below. This allows the same pipeline to be reused across different asset classes and experimental scenarios without modifying the processing logic.

### 1. Asset Class Selection

Select the asset class to be processed:

- **Overhead Conductors** — `overhead_conductors`
- **Transformers** — `transformers`
- **Underground Cables** — `underground_cables` (configuration pending)

The selected asset determines which configuration files and dataset directory the pipeline uses.

### 2. Dataset Selection

Specify the dataset scenario to evaluate.

For example:

`completion0.9-noise0.05`

This represents a dataset with:

- **Completion rate:** 90%
- **Missingness rate:** 10%
- **Noise level:** 5%

Each scenario requires two input files:

| File | Description |
|------|-------------|
| Measured Data | Dataset containing missing values and potentially noisy observations |
| Ground Truth | Complete reference dataset containing the corresponding true values |

The ground-truth dataset is used **only for evaluation** and must not be provided to the imputation model during training.

### 3. Configuration Files

The pipeline automatically loads the configuration files associated with the selected asset:

- `data_schema.yaml` — Attribute definitions, data types, constraints, and feature roles.
- `cleaning_rules.yaml` — Data cleaning and preprocessing rules.
- `imputation.yaml` — Imputation attributes and model configurations.
- `pipeline.yaml` — Pipeline execution settings and evaluation protocols.

### 4. Output Directory

Each experiment uses a separate output directory based on the asset class and dataset scenario.

Example:

`outputs/transformers/completion0.9-noise0.05/`

This keeps experimental results organised and supports comparisons across different asset classes, missingness rates, noise levels, and imputation methods.

### 5. Instructions

**Before running the notebook:**

1. Select the appropriate `ASSET`.
2. Specify the dataset scenario using `SCENARIO`.
3. Verify the measured and ground-truth filenames.
4. Check that the configuration and dataset directories exist.
5. Run the configuration cell before executing the remaining pipeline stages.

> **Note:** All subsequent notebook cells use the paths defined here. When switching datasets or asset classes, update this configuration cell and rerun the notebook from the beginning.

In [1]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

from pathlib import Path
import os

# Automatically locate the project root
CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (path / "src" / "__init__.py").is_file()
        and (path / "config").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the pipeline project root.")


# ---------------- ASSET CONFIGURATION ----------------

ASSET = "transformers"

CONFIG_DIR = PROJECT_ROOT / "config" / ASSET

PIPELINE_CONFIG = CONFIG_DIR / "pipeline.yaml"
DATA_SCHEMA_CONFIG = CONFIG_DIR / "data_schema.yaml"
CLEANING_CONFIG = CONFIG_DIR / "cleaning_rules.yaml"
IMPUTATION_CONFIG = CONFIG_DIR / "imputation.yaml"


# ---------------- DATASET CONFIGURATION ----------------

SCENARIO = "completion0.9-noise0.05"

DATA_DIR = PROJECT_ROOT / "data" / ASSET

MEASURED_FILE = f"{SCENARIO}-measured.csv"
TRUTH_FILE = f"{SCENARIO}-truth.csv"

INPUT_CSV = Path(
    os.getenv("OC_NOTEBOOK_INPUT", str(DATA_DIR / MEASURED_FILE))
)

TRUTH_CSV = Path(
    os.getenv("OC_NOTEBOOK_TRUTH", str(DATA_DIR / TRUTH_FILE))
)


# ---------------- OUTPUT CONFIGURATION ----------------

RUN_DIR = Path(
    os.getenv(
        "OC_NOTEBOOK_RUN_DIR",
        str(PROJECT_ROOT / "outputs" / ASSET / SCENARIO)
    )
)


# ---------------- DISPLAY CONFIGURATION ----------------

HOME_DIR = str(Path.home())

print("Project Root:", str(PROJECT_ROOT).replace(HOME_DIR, "~", 1))
print("Asset:", ASSET)
print("Configuration:", str(PIPELINE_CONFIG).replace(HOME_DIR, "~", 1))
print("Measured Data:", str(INPUT_CSV).replace(HOME_DIR, "~", 1))
print("Ground Truth:", str(TRUTH_CSV).replace(HOME_DIR, "~", 1))
print("Output Directory:", str(RUN_DIR).replace(HOME_DIR, "~", 1))

Project Root: ~/Desktop/GitHub/team19-asset-data/pipeline
Asset: transformers
Configuration: ~/Desktop/GitHub/team19-asset-data/pipeline/config/transformers/pipeline.yaml
Measured Data: ~/Desktop/GitHub/team19-asset-data/pipeline/data/transformers/completion0.9-noise0.05-measured.csv
Ground Truth: ~/Desktop/GitHub/team19-asset-data/pipeline/data/transformers/completion0.9-noise0.05-truth.csv
Output Directory: ~/Desktop/GitHub/team19-asset-data/pipeline/outputs/transformers/completion0.9-noise0.05


## 0. Environment Initialization

This section initializes the Python environment and imports the required libraries and reusable pipeline modules.

The notebook uses the project paths and asset configuration defined in **Section 0: Project Configuration**.

### Environment Requirements

- **Python environment:** `pipeline/.pipeline`
- **Source code:** `pipeline/src/`
- **Notebooks:** `pipeline/notebook/`
- **Configuration:** Selected through the Project Configuration cell.

The project root is detected automatically, allowing the notebook to run from the `pipeline/` directory, the `notebook/` directory, or the parent repository directory.

> **Note:** Run the Project Configuration cell before executing this section. Any changes to asset classes, datasets, configuration files, or output directories should be made in Section 0.

In [2]:
# ============================================================
# PIPELINE INITIALIZATION
# ============================================================

import json
import os
import shutil
import sys

import pandas as pd
from IPython.display import display

# Make the project modules importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import __version__

from src.notebook import (
    display_path,
    key_values,
    list_outputs,
    missingness_before_after,
    notebook_paths,
    show_figures,
    show_images,
    table,
)

# Display environment information with shortened paths
HOME_DIR = str(Path.home())

print("Pipeline root:", str(PROJECT_ROOT).replace(HOME_DIR, "~", 1))
print("Python executable:", str(sys.executable).replace(HOME_DIR, "~", 1))
print("Pipeline version:", __version__)

Pipeline root: ~/Desktop/GitHub/team19-asset-data/pipeline
Python executable: ~/Desktop/GitHub/team19-asset-data/pipeline/.pipeline/bin/python
Pipeline version: 1.1.0


Change the paths below when moving from the synthetic sample to approved partner data. The notebook distinguishes four things that are easy to conflate:

| Name | What it is | Default |
|---|---|---|
| `INPUT_CSV` | the records to complete (operational run, sections 1-6) | synthetic sample with injected data-quality defects |
| `TRUTH_CSV` | a **complete reference** with every model attribute known (evaluation mode, section 7) | synthetic complete reference |
| evaluation partitions | fit / validation / test, drawn by feeder inside the evaluation stage and written to `evaluation/partition_manifest.csv` | from `config/transformers/pipeline.yaml` |

Environment variables `OC_NOTEBOOK_INPUT`, `OC_NOTEBOOK_TRUTH`, `OC_NOTEBOOK_RUN_DIR` override these without editing the notebook (the automated test uses them). `OC_NOTEBOOK_OVERRIDES` may hold a JSON object of hyperparameter overrides for development runs; leave it unset for a release run so the approved `config/transformers/imputation.yaml` values apply.

In [ ]:
PATHS = notebook_paths(PROJECT_ROOT, {
    "input": "data/transformers/completion0.9-noise0.05-measured.csv",
    "truth": "data/transformers/completion0.9-noise0.05-truth.csv",
    "run_dir": "outputs/notebook_demo/transformers",
})
INPUT_CSV, TRUTH_CSV, RUN_DIR = (PATHS[k] for k in ("input", "truth", "run_dir"))

# Development-only hyperparameter overrides (recorded in run_metadata.json); {} for a release run.
OVERRIDES = json.loads(os.environ.get("OC_NOTEBOOK_OVERRIDES", "{}"))
# Set to a name (e.g. "J. Smith, SAPN") to record approval of the CURRENT configuration content from
# this notebook. None leaves config/transformers/approvals.json untouched.
APPROVE_CONFIGURATION_AS = None
# Start from an empty run directory so no file from an earlier run can be mistaken for this one.
FRESH_RUN = True

for name, path in PATHS.items():
    print(f"{name:>8}: {display_path(path, PROJECT_ROOT)}")
print("overrides:", OVERRIDES or "none")

### Configuration check and approval state

`ConfigurationManager` parses the four YAML files (`data_schema.yaml`, `cleaning_rules.yaml`, `imputation.yaml`, `pipeline.yaml`), runs the cross-file checks (model attributes in the imputation config must match the schema; cleaning mappings must map onto allowed labels; forbidden predictors must not be model attributes) and compares each file's SHA-256 with `config/transformers/approvals.json`. In `release` mode every stage refuses to run unless all four files are valid and approved at their current content; in `development` mode the state is recorded in the run metadata and the stage continues.

In [ ]:
from src.configuration import ConfigurationManager, load_settings
from src.pipeline import RunPaths, read_metadata

settings = load_settings(PIPELINE_CONFIG)
manager = ConfigurationManager(settings)
check = manager.check()

print("Execution mode:", settings.execution_mode)
print("Configuration valid:", check.is_valid, "| all files approved:", check.all_approved, "| ready:", check.ready)
display(check.states_table())
display(check.findings_table())

In [ ]:
if APPROVE_CONFIGURATION_AS:
    manager.approve_all(by=APPROVE_CONFIGURATION_AS, note="approved from pipeline.ipynb")
    check = manager.check()
    display(check.states_table())
elif not check.ready and settings.execution_mode == "release":
    print("Release mode will refuse to run. Fix the findings above, then approve the files "
          "(oc-pipeline config approve --by <name>, or set APPROVE_CONFIGURATION_AS), "
          "or switch execution.mode to development in config/transformers/pipeline.yaml.")
else:
    print("Configuration state unchanged.")

### What the configuration says

Schema roles, the two **primary targets** (`installation_year`, `material`) inside the seven **model attributes**, the forbidden predictors, the cleaning rule families with their approval state, and the registered imputation methods with the hyperparameters that will be used.

In [ ]:
summaries = manager.summaries(check.loaded)
display(summaries["schema_summary"])
display(summaries["schema_columns"])
display(summaries["cleaning_families"])
display(summaries["imputation_methods"])
display(summaries["imputation_columns"])

schema, rules, config = check.loaded.schema, check.loaded.cleaning_rules, check.loaded.imputation
run = RunPaths(RUN_DIR)
if FRESH_RUN and RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data ingestion

**Purpose:** preserve the original input, record its structure and checksum, and keep every value as the text it arrived as. This stage does *not* coerce numeric values, normalise categories, delete rows, or fill missing values.

In [ ]:
from src.ingestion import ingest_csv

ingestion = ingest_csv(
    input_csv=INPUT_CSV,
    output_dir=RUN_DIR / "ingestion",
    overwrite=True,
)

raw_frame = ingestion.frame

print("Ingestion completed successfully.")
print("Records:", len(raw_frame))
print("Columns:", len(raw_frame.columns))
print("Report:", display_path(ingestion.report_json, PROJECT_ROOT))

In [ ]:
display(raw_frame.head(10))

In [ ]:
report = ingestion.report

summary = pd.DataFrame({
    "Property": [
        "Source file",
        "SHA-256",
        "Total records",
        "Total columns",
        "File size (bytes)",
        "Duplicate columns",
        "Inconsistent rows",
    ],
    "Value": [
        report["source_file"],
        report["sha256"][:16] + "...",
        report["rows"],
        report["columns"],
        report["file_size_bytes"],
        len(report["duplicate_columns"]),
        report["inconsistent_row_count"],
    ],
})

display(summary)

## 2. Data validation

The quality stage maps the file's headers onto the schema (`column_mapping`), reports structural, identifier, numeric, categorical and derived-field issues, and writes `quality_report.csv` and a byte-for-byte `source_records.csv` into the run directory. It **does not fix** invalid values. A clean synthetic reference produces an empty findings table; the dirty sample produces warnings. With `quality.fail_on_error: true` an error-level finding (missing required column, duplicate identifiers, no observed target values) stops the run here, by design.

In [ ]:
from src.pipeline import run_quality

aligned_frame, quality = run_quality(INPUT_CSV, run, settings)
quality_meta = read_metadata(run)["quality"]

print("Renamed columns:", quality_meta["renamed_columns"] or "none")
print("Unknown columns (carried through):", quality_meta["unknown_columns"] or "none")
print("Missing required columns:", quality_meta["missing_required"] or "none")
print(f"Findings: {quality.n_errors} errors, {quality.n_warnings} warnings ->", display_path(run.quality_report, PROJECT_ROOT))
display(table(quality.issues, 40))

## 3. Exploratory data analysis 

Schema-free profiling of the ingested text: blank and null rates per column, tokens that *may* stand for missing values (reported for review, never converted), suggested types, distributions, outliers, missing-value patterns and pairwise relationships. Nothing is written back to the data. Figures are rendered in memory; the schema-aware profile written to `profile/` comes after cleaning (section 4).

In [ ]:
from src.profiling import profile_dataset, render_discovery_figures

profile = profile_dataset(raw_frame)
print({k: v for k, v in profile.summary.items() if k != "note"})
print(profile.summary["note"])
display(profile.tables["column_profile"])
display(profile.tables["potential_missing_tokens"])
display(table(profile.tables["missing_patterns"], 10))
display(table(profile.tables["relationships"], 15))

In [ ]:
show_figures(render_discovery_figures(raw_frame, profile))

## 4. Preprocessing and deterministic recovery

Three steps, each audited:

1. **Cleaning** (`cleaning_rules.yaml`): documented corrections with a per-cell log. Rule families are split into *safe* normalisation (whitespace, missing tokens, label case, spelling variants of an allowed label) and *domain decisions* (code mappings such as `AAC -> Aluminium`, sentinel years, out-of-range values, unknown categories). The domain families run only when `approval.<family>.approved` is true and name who approved them; the shipped `code_mappings` are **not approved** (they need a SAPN decision) and are skipped, which the log and the report state.
2. **Schema-aware profiling** of the cleaned, typed records into `profile/`.
3. **Deterministic recovery and leakage control**: `installation_year` is restored from an observed `asset_age` (`snapshot_year - asset_age`, provenance *derived*, range-checked) before any model runs; then the model attributes are selected and the forbidden predictors are refused. `length_km` is not a model attribute and is carried through unchanged; its gaps stay **unresolved** rather than being filled by an unapproved rule.

The dry run first shows what cleaning *would* change; the following cell applies it.

In [ ]:
from src.quality import plan_changes

plan = plan_changes(aligned_frame, schema, rules)
print({k: plan.summary[k] for k in ("rows_in", "rows_out", "cells_changed", "rows_dropped")})
print("Rule families skipped (domain approval pending):", plan.summary["inactive_families"] or "none")
print("Code mappings pending approval (values left as they are):", plan.summary["pending_code_mappings"] or "none")
display(plan.log.groupby(["rule", "column"]).size().reset_index(name="n_cells"))
display(table(plan.log, 15))

In [ ]:
from src.ingestion import apply_schema_types
from src.pipeline import run_clean, run_profile

cleaned = run_clean(INPUT_CSV, run, settings, frame=aligned_frame)
typed = apply_schema_types(cleaned, schema)
cleaning_meta = read_metadata(run)["cleaning"]
print("Cleaned records:", display_path(run.cleaned, PROJECT_ROOT), "| log:", display_path(run.cleaning_log, PROJECT_ROOT))
print({k: cleaning_meta[k] for k in ("rows_in", "rows_out", "cells_changed", "rows_dropped")})

profile_result = run_profile(run.cleaned, run, settings, frame=typed)
display(profile_result.tables["missingness"])
display(table(profile_result.tables["missing_patterns"], 10))
show_images(sorted(run.profile_dir.glob("*.png")))

In [ ]:
from src.preprocessing import prepare_for_imputation

prepared = prepare_for_imputation(typed, schema, config)   # run_impute repeats this step internally
model_frame = prepared.model_frame
print("Model attributes (fill and predict):", list(model_frame.model_columns))
print("Primary targets:", list(schema.primary_targets))
print("Carried through, never used for prediction:", list(model_frame.passthrough_columns))
print("Forbidden predictors (schema):", list(schema.forbidden_predictors))
display(prepared.recovery.log)
display(missingness_before_after(typed, prepared.frame, model_frame.model_columns))

## 5. Choose imputation method

The registry offers interchangeable methods behind one interface (`fit` / `transform` / `fit_transform` returning imputed values, per-cell diagnostics and metadata): `median_mode` (reference), `missforest` (original transductive MissForest) and `predictive_missforest` (fit/transform adaptation with frozen sequential replay). `METHOD` defaults to `imputation.method` in `pipeline.yaml`; hyperparameters come from `imputation.yaml` (`methods.<name>` plus `selected_hyperparameters`, written by the tuning experiment) and are recorded in `run_metadata.json`.

`run_impute` fits the method on the prepared records and fills their gaps, runs the reference method on the same records, saves a persistable model with provenance, then hands the result to postprocessing (section 6).

In [ ]:
from src.imputation import available_methods, resolve_hyperparameters

METHOD = settings.method            # or any of available_methods()
print("Registered methods:", available_methods())
print("Selected:", METHOD, "| reference:", settings.reference_method)
display(key_values(resolve_hyperparameters(config, METHOD, OVERRIDES or None)))

In [ ]:
from src.pipeline import run_impute

output = run_impute(run.cleaned, run, settings, method=METHOD, overrides=OVERRIDES or None, frame=typed)
imputation_meta = read_metadata(run)["imputation"]

print("Cells imputed:", imputation_meta["cells_imputed"], imputation_meta["cells_imputed_by_column"])
print("Model saved:", display_path(imputation_meta["model_path"], PROJECT_ROOT) if imputation_meta["model_path"] else "no (transductive method)")
print("Method metadata:", {k: v for k, v in imputation_meta["method_metadata"].items() if not isinstance(v, (dict, list))})
display(table(output.diagnostics, 10))
if run.convergence.is_file():
    display(pd.read_csv(run.convergence))

### Reusing a saved model

A saved `predictive_missforest` model carries a provenance sidecar (fitting rows, identifiers, column roles, constraints, hash). Before `oc-pipeline apply` uses it on new records, `check_compatibility` compares it with the current configuration and input: different model columns are an error; different constraints or unseen category labels are warnings.

In [ ]:
from src.imputation import check_compatibility, load_imputer, read_provenance

if run.model.is_file():
    saved = load_imputer(run.model)
    provenance = read_provenance(run.model)
    compatibility = check_compatibility(
        run.model, saved,
        numeric_columns=model_frame.numeric_columns, categorical_columns=model_frame.categorical_columns,
        constraints=config.constraints, frame=prepared.frame,
    )
    print("Fitted on", provenance.get("n_fitting_rows"), "rows at", provenance.get("created_utc"))
    print("Compatibility errors:", compatibility["errors"] or "none")
    print("Compatibility warnings:", compatibility["warnings"] or "none")
else:
    print("No persisted model for this method (transductive); use `impute` on each new table.")

## 6. Postprocessing and cell provenance

After imputation the derived column is filled or recomputed from its (possibly imputed) source, integer columns are exported as integers, and every cell of `final_imputed.csv` is classified: **observed** (left exactly as cleaned), **derived** (deterministic rule), **imputed** (the method) or **unresolved** (still missing). The integrity checks confirm that no observed value changed and that constraints hold; a failure raises before any file is reported as final.

In [ ]:
from src.ingestion import load_records

final = apply_schema_types(load_records(run.imputed, schema).frame, schema)
provenance_summary = pd.read_csv(run.provenance_summary)
cells = pd.read_csv(run.cell_provenance)

print("Provenance totals:", imputation_meta["provenance"]["totals"])
print("Unresolved columns:", imputation_meta["provenance"]["unresolved_columns"] or "none")
print("Integrity checks passed:", imputation_meta["integrity_passed"])
display(provenance_summary)
display(missingness_before_after(typed, final, final.columns))

In [ ]:
display(table(cells[cells["status"] == "imputed"], 10))
display(table(cells[cells["status"] == "derived"], 10))
display(pd.read_csv(run.derived_log))
display(final.head(10))

## 7. Evaluation and diagnostics

Two different questions, kept apart:

**Operational checks** (`run_validate`) need no ground truth: integrity, distribution shift between observed and imputed values, low-confidence cells, and the *same-table reconstruction check* (hide 10 % of currently observed cells, refit on the masked table, score the hidden cells). The reconstruction check tells how well *known* values are reconstructed on this table; it says nothing about the originally missing cells, which have no truth.

**Evaluation mode** (`run_evaluate`) needs a complete reference. Records are divided into fit / validation / test with the asset's configured partition policy, the measured file's originally missing cells are scored where the truth is known, and only the **test** partition is scored. `unseen_records` fits on the fit partition and transforms the test partition (the operational situation for a persistable method); `transductive` runs `fit_transform` on fit + test together. The validation partition is reserved for tuning and is never scored here. On synthetic data these figures describe the generator, not SAPN's population.

In [ ]:
from src.pipeline import run_validate

validation = run_validate(run, settings)
vdir = run.validation_dir
print("Integrity passed:", validation["integrity_passed"], "| failed checks:", validation["integrity_failed_checks"])
display(pd.read_csv(vdir / "integrity_checks.csv"))
display(pd.read_csv(vdir / "distribution_checks.csv"))
if (vdir / "reconstruction_summary.csv").is_file():
    display(pd.read_csv(vdir / "reconstruction_summary.csv"))
if (vdir / "low_confidence_cells.csv").is_file():
    print("Low-confidence cells listed:", validation["confidence"])
    display(table(pd.read_csv(vdir / "low_confidence_cells.csv"), 10))
show_images(sorted((vdir / "figures").glob("*.png")))

In [ ]:
from src.pipeline import run_evaluate

evaluation = run_evaluate(run, settings, truth_csv=TRUTH_CSV, input_csv=run.cleaned, overrides=OVERRIDES or None)
edir = run.evaluation_dir

print("Truth:", display_path(TRUTH_CSV, PROJECT_ROOT), "| measured input (cleaned):", display_path(run.cleaned, PROJECT_ROOT))
print("Mask source:", evaluation["mask_source"], "| partitions:", evaluation["partitions"], "| strategy:", evaluation["partition_strategy"])
print("Methods and hyperparameters:", evaluation["hyperparameters"])
eval_summary = pd.read_csv(edir / "summary.csv")
display(eval_summary[eval_summary["variable"].isin(schema.primary_targets)])
display(eval_summary)
display(table(pd.read_csv(edir / "per_class.csv"), 20))
show_images(sorted((edir / "figures").glob("*.png")))

## 8. Final export and report

The run directory already holds the deliverables; this section renders the single-file HTML report over all of them and lists what to hand over:

- `final_imputed.csv` - completed records in the source column order, integers as integers;
- `cell_provenance.csv` / `provenance_summary.csv` - what was derived, imputed or left unresolved, and by which rule or method;
- `cleaning_log.csv`, `recovery_log.csv`, `derived_fields_log.csv` - every transformation, one row per cell or rule;
- `run_metadata.json` - configuration state and hashes, settings, hyperparameters, per-stage summaries, versions;
- `validation/` and `evaluation/` - the checks and (synthetic) evaluation tables and figures;
- `model.joblib` + `.provenance.json` - the fitted persistable model, if the method supports it;
- `report.html` - all of the above, readable.

The notebook is a readable controller for reusable Python modules, not the only way to execute the system: `oc-pipeline all --asset transformers --measured <csv> --truth <csv> --out <dir>` produce the same files.

In [ ]:
from src.pipeline import run_report

report_path = run_report(run, settings)
meta = read_metadata(run)
print("Report:", display_path(report_path, PROJECT_ROOT))
print("Run metadata:", display_path(run.metadata, PROJECT_ROOT))
print("Package", meta["package_version"], "| configuration ready:", meta["configuration"]["ready"], "| execution mode:", meta["configuration"]["execution_mode"])
display(list_outputs(RUN_DIR, PROJECT_ROOT))